In [1]:
%load_ext jupyter_black

In [2]:
import warnings

warnings.filterwarnings("ignore")

In [3]:
import os
import gc
from dotenv import load_dotenv

import torch
import numpy as np
import pandas as pd
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from sentence_transformers import SentenceTransformer

In [4]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
device

'mps'

In [5]:
client = QdrantClient("localhost", port=6333)

load_dotenv()
hf_token = os.getenv("HF_TOKEN")
model_name = "intfloat/multilingual-e5-small"
model = SentenceTransformer(model_name, token=hf_token, device=device)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6975.90it/s]
BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
if not client.collection_exists("legal_docs"):
    client.create_collection(
        collection_name="legal_docs",
        vectors_config=VectorParams(size=384, distance=Distance.COSINE),
    )

In [7]:
chunks = pd.read_json("../data/processed/chunks/trafik_kanunu.jsonl", lines=True)

In [8]:
def embed_in_batches(model, texts, batch_size=16):
    all_embeddings = []
    prefixed = [f"query: {t}" for t in texts]
    for i in range(0, len(prefixed), batch_size):
        batch = prefixed[i : i + batch_size]
        embeddings = model.encode(
            batch, convert_to_numpy=True, normalize_embeddings=True
        )
        all_embeddings.append(embeddings)
        if device == "mps":
            torch.mps.empty_cache()  # clear MPS cache each batch
        gc.collect()
    return np.vstack(all_embeddings)

In [9]:
texts = chunks["text"].tolist()
metadatas = chunks["metadata"].tolist()

embeddings = embed_in_batches(model, texts, batch_size=16)

points = [
    PointStruct(
        id=i,
        vector=emb.tolist(),
        payload={
            "text": text_val,  # Add the actual text here
            **metadata_val,  # Unpack madde_no and char_count
        },
    )
    for i, (text_val, metadata_val, emb) in enumerate(zip(texts, metadatas, embeddings))
]

UPSERT_BATCH = 100
for i in range(0, len(points), UPSERT_BATCH):
    client.upsert(collection_name="legal_docs", points=points[i : i + UPSERT_BATCH])